# Stress Prediction — 근접중복 하드 오버라이드 + v34(gender제거)

EDA에서 test의 12.8%가 train과 distance<0.1(사실상 동일), 일부는 distance=0(완전 동일)이었습니다.
지금 v34는 이런 "확실한 매칭"에도 pair 비중 0.28만 반영합니다.

**이번엔 아주 가까운 매칭(threshold 이하)만 예외적으로 그 이웃의 타겟값을 그대로 쓰고,
나머지 대부분은 오늘 검증된 v34(gender 제거, CV 0.1477) 그대로 둡니다.**
전체를 바꾸는 게 아니라 일부만 건드리는 보수적인 방식이라, 실패해도 피해가 제한적입니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from scipy.spatial import cKDTree

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def v34_prediction_no_gender(train_features, val_features, target) -> np.ndarray:
    train_no_gender = train_features.drop(columns=["gender"])
    val_no_gender = val_features.drop(columns=["gender"])
    weighted_train = add_feature_copies(train_no_gender)
    weighted_val = add_feature_copies(val_no_gender)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    tree_pred = np.quantile(tree_predictions, TREE_QUANTILE, axis=1)

    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        median = np.nanmedian(train_features[column].to_numpy(dtype=float))
        tr = np.nan_to_num(train_features[column].to_numpy(dtype=float), nan=median)
        va = np.nan_to_num(val_features[column].to_numpy(dtype=float), nan=median)
        sorted_tr = np.sort(tr)
        train_rank[column] = np.searchsorted(sorted_tr, tr, side="right") / len(tr)
        val_rank[column] = np.searchsorted(sorted_tr, va, side="right") / len(tr)
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    pair_pred = np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)

    return np.clip(np.round((1 - PAIR_WEIGHT) * tree_pred + PAIR_WEIGHT * pair_pred, 2), 0.0, 1.0)


def nearest_neighbor_lookup(train_raw, val_raw, target, numeric_cols):
    train_numeric = train_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    val_numeric = val_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    val_scaled = scaler.transform(val_numeric)
    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(val_scaled, k=1)
    nearest_target = target.to_numpy(dtype=float)[nearest_idx]
    return nearest_dist, nearest_target


def hybrid_prediction(train_raw, val_raw, train_features, val_features, target, numeric_cols, threshold):
    base_pred = v34_prediction_no_gender(train_features, val_features, target)
    nearest_dist, nearest_target = nearest_neighbor_lookup(train_raw, val_raw, target, numeric_cols)
    override_mask = nearest_dist < threshold
    result = base_pred.copy()
    result[override_mask] = np.clip(np.round(nearest_target[override_mask], 2), 0.0, 1.0)
    return result, override_mask


## 5-Fold CV — threshold별 비교 (0=오버라이드 없음=candidate B와 동일)

In [2]:

train = pd.read_csv(DATA_DIR / "train.csv")
raw_X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)
numeric_cols = train.drop(columns=[TARGET, ID_COLUMN]).select_dtypes(include=["number"]).columns.tolist()

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
THRESHOLDS = [0.0, 0.02, 0.05, 0.1, 0.2, 0.3]

oof_by_threshold = {t: np.zeros(len(y)) for t in THRESHOLDS}
override_counts = {t: 0 for t in THRESHOLDS}

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
    val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    train_raw_fold = train.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = train.iloc[val_idx].reset_index(drop=True)

    base_pred = v34_prediction_no_gender(train_feat, val_feat, y_train)
    nearest_dist, nearest_target = nearest_neighbor_lookup(train_raw_fold, val_raw_fold, y_train, numeric_cols)

    for t in THRESHOLDS:
        if t == 0.0:
            oof_by_threshold[t][val_idx] = base_pred
            continue
        override_mask = nearest_dist < t
        override_counts[t] += override_mask.sum()
        pred = base_pred.copy()
        pred[override_mask] = np.clip(np.round(nearest_target[override_mask], 2), 0.0, 1.0)
        oof_by_threshold[t][val_idx] = pred

    print(f"Fold {fold} done")

rows = []
for t in THRESHOLDS:
    mae = mean_absolute_error(y, oof_by_threshold[t])
    rows.append({"threshold": t, "n_overridden": override_counts[t], "cv_mae": mae})

result_df = pd.DataFrame(rows).sort_values("cv_mae")
print()
display(result_df)
print()
print("(참고) v34(gender 제거) 오버라이드 없음: threshold=0.0 행과 동일해야 함")


Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done



,threshold,n_overridden,cv_mae
4,0.20,572,0.147633
5,0.30,598,0.147633
3,0.10,346,0.147653
0,0.00,0,0.147787
1,0.02,70,0.147787
2,0.05,126,0.147787



(참고) v34(gender 제거) 오버라이드 없음: threshold=0.0 행과 동일해야 함


## 최적 threshold로 최종 제출 파일 생성

In [3]:

BEST_THRESHOLD = 0.0  # 결과 1위로 수정 (0.0이면 candidate B와 완전히 동일)

test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
raw_X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)

final_base = v34_prediction_no_gender(raw_X, raw_X_test, y)
if BEST_THRESHOLD > 0:
    final_dist, final_nearest_target = nearest_neighbor_lookup(train, test, y, numeric_cols)
    override_mask = final_dist < BEST_THRESHOLD
    final_prediction = final_base.copy()
    final_prediction[override_mask] = np.clip(np.round(final_nearest_target[override_mask], 2), 0.0, 1.0)
    print(f"오버라이드된 test 샘플 수: {override_mask.sum()} / {len(test)}")
else:
    final_prediction = final_base

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_hybrid_override_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_hybrid_override_0806.csv
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.51
4,TEST_0004,0.53
5,TEST_0005,0.47
6,TEST_0006,0.78
7,TEST_0007,0.48
8,TEST_0008,0.57
9,TEST_0009,0.88
